# PrivHSD: Privacy-preserving Hate Speech Detection

## Interactive Demo Notebook

Council of Europe Democracy Hackathon 2026

This notebook demonstrates the PrivHSD system: training an identity-disentangled differentially private transformer for hate speech detection that operates agnostically to author identity.

In [ ]:
# Setup
import sys
sys.path.append('..')

import torch
import numpy as np
import logging
logging.basicConfig(level=logging.INFO)

from transformers import AutoTokenizer
from src.model import PrivHSDModel
from src.data_utils import load_jigsaw_dataset, get_dataloaders
from src.train import PrivHSDTrainer
from src.evaluate import evaluate_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
# Load tokenizer and dataset
tokenizer = AutoTokenizer.from_pretrained('albert-base-v2')

train_dataset, val_dataset, test_dataset, n_authors = load_jigsaw_dataset(
    data_dir='../data/jigsaw',
    tokenizer=tokenizer,
    max_length=256,
    sample_size=500,  # Small sample for demo
)

train_loader, val_loader, test_loader = get_dataloaders(
    train_dataset, val_dataset, test_dataset,
    batch_size=8,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Author classes: {n_authors}")

In [ ]:
# Initialize model with adversarial disentanglement
model = PrivHSDModel(
    model_name='albert-base-v2',
    num_hate_classes=2,
    num_authors=n_authors,
    adversarial_alpha=0.5,
    disentanglement_weight=0.3,
    model_type='albert',
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Initialize trainer with DP-SGD
trainer = PrivHSDTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    learning_rate=2e-5,
    target_epsilon=8.0,
    dp_enabled=True,
    adversarial_alpha=0.5,
    disentanglement_weight=0.3,
    device=device,
    output_dir='../models/checkpoints/demo',
)

print("Training setup complete.")

In [ ]:
# Train (2 epochs for demo)
results = trainer.train(
    num_epochs=2,
    use_adversarial=True,
    eval_every=1,
)

print(f"Final epsilon: {results['final_epsilon']:.2f}")
print(f"Best val F1: {results['best_val_f1']:.4f}")

In [ ]:
# Evaluate on test set
eval_result = evaluate_model(model, test_loader, device)

print("=" * 50)
print("TEST SET EVALUATION")
print("=" * 50)
print(f"F1 Score:     {eval_result.utility.f1_score:.4f}")
print(f"Accuracy:     {eval_result.utility.accuracy:.4f}")
print(f"AUC-ROC:      {eval_result.utility.roc_auc:.4f}")
print(f"Precision:    {eval_result.utility.precision:.4f}")
print(f"Recall:       {eval_result.utility.recall:.4f}")
print(f"MCC:          {eval_result.utility.mcc:.4f}")
print(f"Privacy (ε):  {results['final_epsilon']:.2f}")

In [ ]:
# Interactive hate speech detection
def detect_hate(text):
    """Detect hate speech in a given text."""
    model.eval()
    with torch.no_grad():
        encodings = tokenizer(
            text, truncation=True, padding='max_length',
            max_length=256, return_tensors='pt',
        )
        input_ids = encodings['input_ids'].to(device)
        attention_mask = encodings['attention_mask'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = outputs['hate_probs'][0].cpu().numpy()
        
    hate_prob = probs[1]  # Probability of hateful class
    label = "HATE SPEECH" if hate_prob > 0.5 else "NON-HATE"
    
    print(f"Text: {text[:100]}..." if len(text) > 100 else f"Text: {text}")
    print(f"Prediction: {label}")
    print(f"Confidence: {hate_prob:.4f}")
    print()
    return label, hate_prob

# Test examples
test_texts = [
    "This is a normal comment about the weather today.",
    "You are an ignorant fool who should be banned from this platform.",
    "I disagree with your political观点 but I respect your right to express them.",
    "All [group] are inferior and should not be allowed here.",  # Synthetic hate
    "Great point! I hadn't considered that perspective before.",
]

for text in test_texts:
    detect_hate(text)

In [ ]:
# Privacy audit
from src.attacks import (
    MembershipInferenceAttack,
    AttributeInferenceAttack,
    StylometryReidentificationRisk,
    RepresentationPrivacyAudit,
)

print("=" * 50)
print("PRIVACY AUDIT")
print("=" * 50)

# MIA
mia = MembershipInferenceAttack(attack_type='shadow_model')
member_texts = [train_dataset.texts[i] for i in range(min(50, len(train_dataset)))]
non_member_texts = [test_dataset.texts[i] for i in range(min(50, len(test_dataset)))]
mia_metrics = mia.evaluate(model, member_texts, non_member_texts, tokenizer, device)
print(f"MIA AUC: {mia_metrics.auc:.4f}")

# Stylometry
stylo = StylometryReidentificationRisk(n_authors=n_authors)
if len(test_dataset) >= 50 and test_dataset.author_ids:
    sample_texts = [test_dataset.texts[i] for i in range(min(50, len(test_dataset)))]
    sample_authors = np.array([test_dataset.author_ids[i] for i in range(min(50, len(test_dataset)))])
    stylo_results = stylo.evaluate(model, sample_texts, sample_authors, tokenizer, device)
    print(f"Stylometry (model reps) acc: {stylo_results['model_representation'].accuracy:.4f}")
    print(f"Stylometry (raw text) acc: {stylo_results['raw_text'].accuracy:.4f}")

print("\n✓ Privacy audit complete")